# Does the score predict buying?

The copilot gives each project a 1–10 score per security service. This notebook checks whether those
scores are any good, in three steps:

1. **Snapshot** — score a list of projects today and save the results with a timestamp.
2. **Expert check (now)** — compare the copilot's priority with your own BD judgment, and review where you disagree.
3. **Outcome check (later)** — after 60–90 days, record what projects actually did (announced an audit,
   replied, booked a call) and measure whether high scores predicted it. With enough outcomes, fit
   better scoring weights.

**Why snapshots?** Scoring a project today and comparing it with what it did *last month* leaks the answer:
a fresh audit already lowers the audit score. Only outcomes that happen *after* the snapshot count.

**Before running:** start the copilot (`npm run dev` in the project folder). Snapshots are saved in
`eval/snapshots/`, which is git-ignored because briefs contain people's names. Clear outputs
(*Edit → Clear all outputs*) before committing this notebook.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import evaluate as ev

API_URL = "http://127.0.0.1:3000/api/analyze"   # the local copilot
SNAPSHOT = Path("snapshots") / f"{datetime.now(timezone.utc):%Y-%m-%d}.csv"   # or an older file to re-evaluate
LIMIT = None          # e.g. 5 for a quick trial run

projects = pd.read_csv("projects.csv")
outcomes = pd.read_csv("outcomes.csv")
print(f"{len(projects)} projects, {projects['bd_priority'].notna().sum()} with your label, {len(outcomes)} outcomes recorded")

## 1. Snapshot

Analyses each project one at a time (about 5–15 s each) and appends to the snapshot file.
It's resumable: rerun the cell and finished projects are skipped.

In [ ]:
inputs = projects["input"].tolist()[:LIMIT]
snap = ev.run_snapshot(inputs, SNAPSHOT, url=API_URL)
snap.head()

### Data health

Scores built on thin data deserve less trust. This shows how much the copilot could actually see.

In [ ]:
ev.health(snap)

In [ ]:
top = ev.top_service(snap)
top[["input", "name", "service", "final", "priority", "confidence", "need", "gap", "timing", "pay", "reach", "cap"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
scored = snap.dropna(subset=["service"])
scored["final"].plot.hist(bins=range(1, 12), ax=ax[0], rwidth=0.85, title="All service scores")
scored.groupby("service")["final"].mean().sort_values().plot.barh(ax=ax[1], title="Average score by service")
plt.tight_layout()

## 2. Expert check

Fill `bd_priority` in `projects.csv` with your own call for each project — **A** (pitch now),
**B** (worth a look), **C** (not now) — and optionally `bd_service` (the service you'd lead with:
`smart_contract_audit`, `protocol_audit`, `tokenomics_review`, `pentest`, `soc2`).
Label **before** looking at the copilot's scores, or you'll anchor on them.

The disagreements are the useful part: each one is either a gap in the copilot's data or rules,
or something you hadn't spotted.

In [ ]:
projects = pd.read_csv("projects.csv")   # reload after editing
agree = ev.expert_agreement(snap, projects)
if agree["n"] == 0:
    print("No labels yet — fill bd_priority in projects.csv and rerun this cell.")
else:
    print(f"Projects compared: {agree['n']}")
    print(f"Same tier: {agree['tier_match_rate']:.0%}   within one tier: {agree['within_one_tier']:.0%}   rank correlation: {agree['spearman']:.2f}")
    if "service_match_rate" in agree:
        print(f"Same lead service: {agree['service_match_rate']:.0%}")
    display(agree["confusion"])
    display(agree["disagreements"][["input", "name", "service", "final", "priority", "bd_priority", "bd_service", "notes"]])

## 3. Outcome check

Add a row to `outcomes.csv` whenever a project in the snapshot does something that counts as buying
(or a step towards it):

| column | example |
|---|---|
| `input` | `pendle.finance` (same as in projects.csv) |
| `service` | `pentest`, or `*` for anything not service-specific (e.g. replied to outreach) |
| `event_date` | `2027-01-15` |
| `kind` | `audit_announced`, `replied`, `meeting`, `won` |
| `source_url` | link to the announcement or your CRM record |

Rows whose 90-day window hasn't closed and have no outcome yet are **pending** and left out —
we don't know the answer yet.

In [ ]:
WINDOW_DAYS = 90
outcomes = pd.read_csv("outcomes.csv")
labelled = ev.label_outcomes(snap, outcomes, window_days=WINDOW_DAYS)
print(f"Rows: {len(labelled)}   outcomes: {labelled['outcome'].sum()}   pending: {labelled['pending'].sum()}")
m = ev.outcome_metrics(labelled, k=10)
m

**How to read it**
- **AUC** — the chance a project that bought scored higher than one that didn't. 0.5 = coin flip, 0.7+ = useful, 0.8+ = strong.
- **precision@10** — of the 10 highest scores, how many led to an outcome. Compare with **base_rate**.
- **Hit rate by band** — should rise with the score. If 6–7 converts as well as 8–10, the top tier isn't earning its place.

In [ ]:
if "hit_rate_by_band" in m:
    m["hit_rate_by_band"]["mean"].plot.bar(title="Outcome rate by score band", ylabel="share with an outcome", rot=0)

## 4. Tune the weights

Logistic regression on the five sub-scores shows which ones actually predict outcomes. Cross-validated
AUC compares current weights with fitted ones. Only switch if the fitted weights are clearly better
**and** there are enough outcomes (20+ positives) — otherwise it's fitting noise.

In [ ]:
fit = ev.fit_weights(labelled)
fit

In [ ]:
if "suggested_weights" in fit:
    print(ev.weights_yaml(fit["suggested_weights"]))
    print("\nCurrent:", ev.CURRENT_WEIGHTS)